In [1]:
# 初始化模型

from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
import os
from dotenv import load_dotenv
from pydantic import SecretStr


load_dotenv(override=True)
# 取消 socks5 代理，避免 httpx 报错 missing socksio
os.environ.pop("all_proxy", None)
os.environ.pop("ALL_PROXY", None)
OPENAI_API_KEY = os.getenv("DEEPSEEK_API_KEY", "")
OPENAI_API_BASE = os.getenv("DEEPSEEK_BASE_URL", "")

# 初始化 ChatOpenAI 模型，指定使用的模型为 'gpt-4o-mini'
model = ChatOpenAI(model="deepseek-chat", api_key=SecretStr(OPENAI_API_KEY), base_url=OPENAI_API_BASE)


/Users/deer/workspace/jk_code/openai-quickstart/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# 导入依赖库
import os
import bs4
from dotenv import load_dotenv
from pydantic import SecretStr

from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.prompts import ChatPromptTemplate

load_dotenv(override=True)
os.environ.pop("all_proxy", None)
os.environ.pop("ALL_PROXY", None)

USER_AGENT environment variable not set, consider setting it to identify your requests.


In [3]:
bs4_strainer = bs4.SoupStrainer("main")
loader = WebBaseLoader(
    web_paths=(
        "https://api-docs.deepseek.com/zh-cn/",
        "https://api-docs.deepseek.com/zh-cn/guides/thinking_mode",
        "https://api-docs.deepseek.com/zh-cn/guides/tool_calls",
        "https://api-docs.deepseek.com/zh-cn/quick_start/pricing",
        "https://api-docs.deepseek.com/zh-cn/guides/multi_round_chat",
    ),
    bs_kwargs={"parse_only": bs4_strainer},
)
docs = loader.load()
print(f"加载文档数量: {len(docs)}")

加载文档数量: 5


In [4]:
# 文档分割
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, chunk_overlap=200, add_start_index=True
)
all_splits = text_splitter.split_documents(docs)
print(f"分割后块数量: {len(all_splits)}")

分割后块数量: 35


In [5]:
# 存储嵌入（使用本地中文 Embedding 模型）
# 设置离线模式，避免网络请求
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"

# 使用本地缓存路径加载模型
model_path = os.path.expanduser("~/.cache/huggingface/hub/models--shibing624--text2vec-base-chinese/snapshots/183bb99aa7af74355fb58d16edf8c13ae7c5433e")
embedding = HuggingFaceEmbeddings(model_name=model_path)

vectorstore = Chroma.from_documents(
    documents=all_splits,
    embedding=embedding
)
print(f"向量数据库中的文档数量: {vectorstore._collection.count()}")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 10439.07it/s]
Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


向量数据库中的文档数量: 35


In [6]:
# 创建检索器
retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 6})
print("检索器创建成功，返回 top-6 个相关文档")

检索器创建成功，返回 top-6 个相关文档


In [7]:
# prompt 模板（优化版）
template = """你是一个专业的 DeepSeek API 技术文档助手。请基于以下上下文内容回答问题。

## 回答要求：
1. **准确性优先**：只使用上下文中的信息，不要编造或推测
2. **结构化输出**：使用清晰的标题和列表组织答案
3. **完整性**：如果上下文包含多个相关点，请全部涵盖
4. **诚实原则**：如果上下文没有足够信息，请明确说明"根据现有文档无法回答此问题"

## 上下文内容：
{context}

## 问题：
{question}

## 回答："""

prompt = ChatPromptTemplate.from_template(template)

In [8]:
# 构建 RAG Chain
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | model
    | StrOutputParser()
)

In [10]:
# 测试
questions = [
    "DeepSeek 的思考模式是什么？如何启用？",
    "DeepSeek 支持哪些模型？它们的价格是多少？",
    "如何在 DeepSeek 中使用 Tool Calls？"
]

for i, question in enumerate(questions, 1):
    print(f"\n{'='*50}")
    print(f"问题 {i}: {question}")
    print(f"{'='*50}")
    for chunk in rag_chain.stream(question):
        print(chunk, end="", flush=True)
    print()


问题 1: DeepSeek 的思考模式是什么？如何启用？
# DeepSeek 思考模式详解

## 1. 什么是思考模式？

DeepSeek 模型支持思考模式，即**在输出最终回答之前，模型会先输出一段思维链内容（reasoning_content），以提升最终答案的准确性**。

## 2. 如何启用思考模式？

### 默认状态
- 思考模式**默认打开**，且 `effort` 默认为 `high`。

### 不同 API 格式的参数设置

| 控制类型 | OpenAI 格式 | Anthropic 格式 | Responses API 格式 |
|---------|-------------|----------------|-------------------|
| 思考模式开关 | `{"thinking": {"type": "enabled/disabled"}}` | `{"reasoning": {"effort": "none/low/high/max"}}` (none 表示关闭思考模式) | 同左 |
| 思考强度控制 | `{"reasoning_effort": "low/high/max"}` | `{"output_config": {"effort": "low/high/max"}}` | 同左 |

### 实际映射说明
用户设置的 `effort` 与模型实际映射关系如下：
- **deepseek-v4-flash**：`low→low`，`high→high`，`xhigh→high`，`max→max`
- **deepseek-v4-pro**：`low→low`，`high→high`，`xhigh→high`，`max→max`（注：deepseek-v4-pro 的实际映射将于 2026 年 8 月初更新）

### OpenAI SDK 调用示例

```python
response = client.chat.completions.create(
    model="deepseek-v4-pro",
    # ... 
    reasoning_effort="high",
    extra_body={"thinking": {"type": "enabled"}}
)
``